# Тетрадка пересечений ИНН ДР vs CRM

Цель:
1) Посчитать количество уникальных ИНН в выгрузке ДР (`default_data.pkl`).
2) Посчитать пересечения между уникальными ИНН ДР и CRM.
3) Дать разбивку количества ИНН из пересечения по сегментам.

Логика CRM-фильтров:
- источники как в `build_dataset.ipynb`: `Н2.0`, `Справочник CRM-системы`, `CRM-система`;
- период: `01.01.2023 - 31.12.2025`;
- сегменты: маппинг `SEGMENT_MAP`, но немаппированные `X_AREA_RESP` сохраняются (не отбрасываются).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [ ]:
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

# Строгие пути как в build_dataset.ipynb
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
DR_PATH = Path(f"{DATA_DIR}/default_data.pkl")
CRM_PATH = Path(f"{DATA_DIR}/crm_last_version.csv")


def normalize_inn(s):
    """Нормализация ИНН с защитой от потери ведущих нулей."""
    if pd.isna(s):
        return None

    s = str(s).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None

    if s.endswith(".0"):
        s = s[:-2]

    digits = "".join(ch for ch in s if ch.isdigit())
    if not digits:
        return None

    if len(digits) in (10, 12):
        return digits
    if len(digits) < 10:
        return digits.zfill(10)
    if len(digits) < 12:
        return digits.zfill(12)

    return digits


def mode_or_none(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else None


In [ ]:
# 1) Уникальные ИНН из выгрузки ДР (с фильтром по периоду, как в build_dataset)
print(f"Источник ДР: {DR_PATH}")

dr_df = pd.read_pickle(DR_PATH)
dr_df = dr_df.astype(str).replace("nan", np.nan)
dr_df.columns = [str(c).strip().lstrip("\ufeff") for c in dr_df.columns]
dr_df = dr_df.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})

if "inn" not in dr_df.columns:
    raise KeyError("В выгрузке ДР не найдена колонка inn")
if "start_date" not in dr_df.columns:
    raise KeyError("В выгрузке ДР не найдена колонка start_date/start")

dr_df["inn"] = dr_df["inn"].apply(normalize_inn)
dr_df["start_date"] = pd.to_datetime(dr_df["start_date"], dayfirst=True, errors="coerce")

dr_filtered = (
    dr_df
    .dropna(subset=["inn", "start_date"])
    .query("@DATE_FROM <= start_date <= @DATE_TO")
    .copy()
)

dr_unique = set(dr_filtered["inn"].unique())

print(f"Уникальных ИНН в ДР (после фильтра по периоду): {len(dr_unique):,}")


In [ ]:
# 2) CRM: фильтрация как в build_dataset, но без жесткого маппинга сегментов
print(f"Источник CRM: {CRM_PATH}")

crm_df = pd.read_csv(CRM_PATH, dtype=str, low_memory=False)
crm_df.columns = [str(c).strip().lstrip("\ufeff") for c in crm_df.columns]

required_cols = ["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "VAL", "X_AREA_RESP"]
missing = [c for c in required_cols if c not in crm_df.columns]
if missing:
    raise KeyError(f"В CRM отсутствуют обязательные колонки: {missing}")

# Для fallback используем VAL.4 (или VAL_4, если колонка переименована)
val4_col = "VAL.4" if "VAL.4" in crm_df.columns else ("VAL_4" if "VAL_4" in crm_df.columns else None)

# Нормализация как в build_dataset
crm_df["IDENTIFICATION_DATE"] = pd.to_datetime(crm_df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
crm_df["X_INN"] = crm_df["X_INN"].apply(normalize_inn)

# 1) Фильтр по источникам
crm_df["VAL"] = crm_df["VAL"].astype(str).str.strip()
crm_df = crm_df[crm_df["VAL"].isin(ALLOWED_SOURCES)].copy()

# 2) Без жесткого маппинга сегментов: сохраняем все сегменты
raw_area_resp = (
    crm_df["X_AREA_RESP"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan, "None": np.nan})
)
crm_df["x_area_resp_raw"] = raw_area_resp
crm_df["segment_base"] = raw_area_resp.map(SEGMENT_MAP)
crm_df["segment"] = crm_df["segment_base"].fillna(raw_area_resp).fillna("без сегмента")

if val4_col is not None:
    crm_df["val4_raw"] = (
        crm_df[val4_col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
else:
    crm_df["val4_raw"] = np.nan

# Нормализация значений VAL.4 к принятому набору сегментов
VAL4_TO_SEGMENT = {
    "Микро": "МкБ",
    "Малый": "МСБ",
    "Средний": "МСБ",
    "Крупный": "ККБ",
}
crm_df["val4_segment"] = crm_df["val4_raw"].map(VAL4_TO_SEGMENT)

# 3) Фильтр по периоду
crm_df = crm_df[
    (crm_df["IDENTIFICATION_DATE"] >= DATE_FROM)
    & (crm_df["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()

# 4) Формирование fp как в build_dataset + отсев без ИНН/ФП
fp = crm_df[["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "segment", "x_area_resp_raw", "val4_raw", "val4_segment"]].copy()
fp = fp.rename(columns={
    "X_INN": "inn",
    "NUMBER_FP_SFP": "fp_type",
    "IDENTIFICATION_DATE": "fp_date",
})
fp = fp.dropna(subset=["inn", "fp_type"]).copy()

crm_filtered = fp.copy()
crm_unique = set(crm_filtered["inn"].dropna().unique())

print(f"Уникальных ИНН в CRM (после фильтров и наличия ФП/СФП): {len(crm_unique):,}")
if val4_col is None:
    print("ВНИМАНИЕ: колонка VAL.4/VAL_4 не найдена, fallback ДРПА -> VAL.4 не применится.")


In [ ]:
# 3) Пересечения множеств ИНН
intersection = dr_unique & crm_unique
dr_only = dr_unique - crm_unique
crm_only = crm_unique - dr_unique

sets_summary = pd.DataFrame([
    {"Метрика": "Уникальных ИНН в ДР", "Количество": len(dr_unique)},
    {"Метрика": "Уникальных ИНН в CRM (после фильтров)", "Количество": len(crm_unique)},
    {"Метрика": "Пересечение ДР ∩ CRM", "Количество": len(intersection)},
    {"Метрика": "Только в ДР", "Количество": len(dr_only)},
    {"Метрика": "Только в CRM", "Количество": len(crm_only)},
])

print("Итоги по множествам ИНН:")
display(sets_summary)


In [ ]:
# 4) Разбивка пересечения по сегментам (количество ИНН)
# Fallback: если сегмент по X_AREA_RESP = ДРПА, берем сегмент из VAL.4
# и нормализуем VAL.4 к принятому набору: МкБ/МСБ/ККБ
crm_intersection = crm_filtered[crm_filtered["inn"].isin(intersection)].copy()

crm_intersection["segment_final"] = crm_intersection["segment"]
mask_drpa = crm_intersection["x_area_resp_raw"].eq("ДРПА")
crm_intersection.loc[mask_drpa, "segment_final"] = crm_intersection.loc[mask_drpa, "val4_segment"]
crm_intersection["segment_final"] = crm_intersection["segment_final"].fillna("без сегмента")

seg_per_inn = (
    crm_intersection
    .groupby("inn", as_index=False)
    .agg(segment=("segment_final", mode_or_none))
)

segment_counts = (
    seg_per_inn
    .assign(segment=lambda d: d["segment"].fillna("без сегмента"))
    .groupby("segment", as_index=False)
    .agg(count_inn=("inn", "nunique"))
    .sort_values("count_inn", ascending=False)
    .reset_index(drop=True)
)

control_total = int(segment_counts["count_inn"].sum())
intersection_total = len(intersection)

drpa_fallback_cnt = int(mask_drpa.sum())

print("Разбивка пересечения ДР ∩ CRM по сегментам (с fallback ДРПА -> VAL.4):")
display(segment_counts)

if mask_drpa.any():
    fallback_quality = (
        crm_intersection.loc[mask_drpa]
        .assign(is_mapped=lambda d: d["val4_segment"].notna())
        .groupby("is_mapped", as_index=False)
        .agg(rows=("inn", "size"))
    )
    print("\nПокрытие fallback по VAL.4 для строк с ДРПА:")
    display(fallback_quality)

print("Проверка консистентности:")
print(f"- Строк пересечения с X_AREA_RESP=ДРПА (кандидаты для fallback): {drpa_fallback_cnt:,}")
print(f"- Сумма по сегментам: {control_total:,}")
print(f"- Размер пересечения: {intersection_total:,}")
print(f"- Совпадает: {control_total == intersection_total}")
